# NLP Day 2 · 실습 3. 서브워드 토큰화와 입력 길이 맞추기

**연계 이론:** 이론 4. 서브워드 토큰화 · Padding · Truncation · Attention Mask

## 1. 이번 실습에서 만들 것

같은 문장을 **띄어쓰기 단위**와 **서브워드 단위**로 토큰화하여 비교하고, 길이가 다른 3개 문장을 최대 길이 12로 처리합니다.

| 구분 | 활동 | 확인할 결과 |
|---|---|---|
| A형 | `자연어 처리를 공부합니다.`의 토큰·ID·길이 비교 | 띄어쓰기와 서브워드 토큰 수의 차이 |
| B형 | `encode_batch()`에서 Padding·Truncation·최대 길이 지정 | `input_ids`, `attention_mask` 모두 `(3,12)` |

**토크나이저 선택:** 가능하면 사전학습 토크나이저 `klue/bert-base`를 사용합니다(최초 1회 다운로드 가능). 해당 토크나이저를 사용할 수 없는 환경에서는 같은 흐름을 실행할 수 있도록 **수업용 작은 한국어 문장 모음으로 학습한 SentencePiece BPE 토크나이저**를 사용합니다. 대체 토크나이저는 사전학습 BERT의 실제 분할 결과와 같지 않습니다.

*필요 라이브러리:* `torch`, `transformers`(사전학습 토크나이저용) 또는 `sentencepiece`(대체용). 프로젝트 환경에서는 `uv add transformers sentencepiece`로 준비할 수 있습니다.

## 2. A형 — 함께 구현하기

### 2.1 토크나이저 준비하기

토크나이저는 **입력 텍스트를 토큰으로 나누고 각 토큰의 ID를 결정**합니다. 이번 실습은 일반 토큰의 분할에 집중하므로 사전학습 토크나이저를 사용할 때도 **특수 토큰 자동 추가는 끕니다.** `[CLS]`, `[SEP]` 등은 이후 BERT 학습에서 따로 살펴봅니다.

In [ ]:
# 토크나이저 준비 → 토큰·ID 비교 → 배치 길이 조정 → 결과 검증
# 한국어 BERT 사전학습 토크나이저 준비

import torch
from transformers import AutoTokenizer

# 실습에서 사용할 한국어 사전학습 토크나이저 지정
TOKENIZER_NAME = "klue/bert-base"

# 사전학습된 WordPiece 토크나이저 로드
# 첫 실행 시 Hugging Face에서 토크나이저 파일 다운로드
tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_NAME,
    use_fast=True
)

# 사용 중인 토크나이저 확인
tokenizer_kind = f"사전학습 토크나이저: {TOKENIZER_NAME}"
print("사용 중인 토크나이저:", tokenizer_kind)


/Users/jian/Documents/module2_cv_nlp/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


사용 중인 토크나이저: 사전학습 토크나이저: klue/bert-base


### 2.2 띄어쓰기와 서브워드의 결과 비교하기

문장을 띄어쓰기로 나눈 결과와 토크나이저로 나눈 결과를 비교합니다. **글자 수, 띄어쓰기 단어 수, 모델 토큰 수는 서로 다를 수 있습니다.**

In [2]:
# 띄어쓰기 분할 → 서브워드 분할 → ID 변환 → 토큰 수 비교
sentence_a = "자연어 처리를 공부합니다."
# 띄어쓰기 단위 토큰 목록
space_tokens = sentence_a.split()
# 토크나이저가 학습한 서브워드 단위 토큰 목록
model_tokens = tokenizer.tokenize(sentence_a)
# 각 서브워드에 대응하는 숫자 ID
model_ids = tokenizer.convert_tokens_to_ids(model_tokens)

print("원본 문장:", sentence_a)
print("띄어쓰기 토큰:", space_tokens)
print("서브워드 토큰:", model_tokens)
print("서브워드 토큰 ID:", model_ids)
print("글자 수:", len(sentence_a))
print("띄어쓰기 토큰 수:", len(space_tokens))
print("모델 토큰 수:", len(model_tokens))

# 서브워드와 토큰 ID가 1:1로 대응하는지 검증
assert len(model_tokens) == len(model_ids)

원본 문장: 자연어 처리를 공부합니다.
띄어쓰기 토큰: ['자연어', '처리를', '공부합니다.']
서브워드 토큰: ['자연', '##어', '처리', '##를', '공부', '##합니다', '.']
서브워드 토큰 ID: [3941, 2051, 4211, 2138, 4244, 11800, 18]
글자 수: 14
띄어쓰기 토큰 수: 3
모델 토큰 수: 7


### 2.3 길이가 다른 문장의 입력 확인하기

아래 세 문장을 각각 토큰화해 길이를 비교합니다. B형에서는 세 문장을 모두 길이 12로 맞춥니다.

In [3]:
# 서로 길이가 다른 세 문장 준비 → 원래 토큰 수 확인
sentences = [
    "안녕하세요.",
    "자연어 처리를 공부합니다.",
    "오늘은 자연어 처리 수업에서 문장을 토큰으로 나누고 서로 다른 길이의 문장을 같은 길이로 맞추는 방법을 공부합니다.",
]
# 모든 문장에 적용할 최대 길이
MAX_LENGTH = 12

# Padding·Truncation 적용 전 문장별 서브워드 개수
for i, sentence in enumerate(sentences, start=1):
    print(f"문장 {i}: 토큰 {len(tokenizer.tokenize(sentence))}개 / 문장: {sentence}")

문장 1: 토큰 4개 / 문장: 안녕하세요.
문장 2: 토큰 7개 / 문장: 자연어 처리를 공부합니다.
문장 3: 토큰 31개 / 문장: 오늘은 자연어 처리 수업에서 문장을 토큰으로 나누고 서로 다른 길이의 문장을 같은 길이로 맞추는 방법을 공부합니다.


## 3. B형 — 직접 완성하기

> **문제:** `encode_batch()` 함수를 완성하여 세 문장을 **최대 길이 12**로 처리하세요. 짧은 문장에는 Padding을 넣고, 긴 문장에는 Truncation을 적용하세요.

토크나이저를 호출할 때 `padding`, `truncation`, `max_length`를 올바르게 지정합니다. 이번에는 **특수 토큰 자동 추가를 끄고**, 결과를 PyTorch Tensor로 받습니다.

In [4]:
def encode_batch(sentences, tokenizer):
    # ★ TODO: 아래 토크나이저 호출을 완성한 뒤 return 하세요.
    # 힌트: padding="max_length", truncation=True, max_length=12
    # return tokenizer(
    #     sentences,
    #     add_special_tokens=False,
    #     padding=...,
    #     truncation=...,
    #     max_length=...,
    #     return_attention_mask=True,
    #     return_tensors="pt",
    # )
    return None

encoded = encode_batch(sentences, tokenizer)
print("B형 입력 생성 완료:", encoded is not None)

B형 입력 생성 완료: False


### 3.1 B형 결과 확인하기

`input_ids`는 각 토큰의 ID를 담고, `attention_mask`는 **실제 토큰을 1**, **Padding 위치를 0**으로 표시합니다. 두 Tensor의 Shape는 모두 `(문장 수, 최대 길이)`가 되어야 합니다.

In [5]:
# 출력 Tensor 확인 → Shape 검증 → 잘림·Padding 수 검증
if encoded is None:
    print("B형 함수를 완성하면 input_ids와 attention_mask가 출력됩니다.")
else:
    # 토큰 ID와 실제 토큰 위치 마스크 분리
    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]
    print("input_ids Shape:", tuple(input_ids.shape))
    print("attention_mask Shape:", tuple(attention_mask.shape))
    print("input_ids:\n", input_ids)
    print("attention_mask:\n", attention_mask)

    # 입력 3개 × 최대 길이 12의 Shape 확인
    assert tuple(input_ids.shape) == (3, 12)
    assert tuple(attention_mask.shape) == (3, 12)

    # 문장별 원본 토큰 수와 최종 실제 토큰·Padding 수 비교
    for idx, sentence in enumerate(sentences):
        before_length = len(tokenizer.tokenize(sentence))
        real_count = int(attention_mask[idx].sum())
        pad_count = MAX_LENGTH - real_count
        print(f"문장 {idx + 1}: 원래 {before_length}토큰 → 실제 {real_count}, Padding {pad_count}")
        assert real_count == min(before_length, MAX_LENGTH)
        assert pad_count == max(0, MAX_LENGTH - before_length)
        # Padding이 존재하면 해당 위치의 마스크가 모두 0인지 확인
        if pad_count > 0:
            assert torch.all(attention_mask[idx, real_count:] == 0)

    print("B형 Shape·Padding·Truncation 확인: 통과")

B형 함수를 완성하면 input_ids와 attention_mask가 출력됩니다.


## 4. 결과 확인과 이번 내용 간단하게 요약

**완료 기준:** `input_ids`와 `attention_mask`의 Shape가 모두 **`(3,12)`**입니다. 실제 토큰의 마스크는 1, Padding의 마스크는 0이며, 긴 문장에서는 최대 길이 뒤쪽의 토큰이 제외됩니다.

결과를 보고 다음을 짧게 설명해 보세요.

- 문장 길이가 같아졌는데도 `attention_mask`가 문장마다 다른 이유는 무엇인가요?
- 긴 문장을 자를 때 원본 정보에는 어떤 변화가 생기나요?

**이번 내용 간단하게 요약**

첫째, 서브워드 토큰화는 문장을 모델이 사용하는 작은 토큰과 ID로 나누는 과정입니다.

둘째, Padding은 짧은 입력을 채우고 Truncation은 긴 입력을 잘라, 여러 문장을 같은 크기의 Tensor로 만듭니다.

셋째, Attention Mask는 실제 토큰과 Padding을 구분합니다. 이후에는 이렇게 준비한 입력을 RNN과 Transformer 등 여러 NLP 모델에 전달하게 됩니다.